### Phase 2 | Rangking Alghoritm Experiment

Pada fase ini akan dilakukan rangking terhadap algoritma TFIDF dengan menggunakan LightGBM Ranker, XGBoost Ranker, SVM Ranker, dan Random Forest

#### Step 1 | Setup Library & Environment

In [1]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost
from dotenv import load_dotenv
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import LinearSVC
import lightgbm as lgb
import xgboost as xgb

try:
    from IPython.display import display
except ImportError:
    display = print

load_dotenv()

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(mlflow_uri)

mlflow.set_experiment("Phase_2_Ranking_Algorithm_Experiment")

print(f"MLflow Connected to: {mlflow_uri}")


MLflow Connected to: https://mlflow.smbgarasibmw.my.id/


#### Step 2 | Load Data & Validasi Query Group

In [2]:
df = pd.read_csv("dataset/Dataset_EDA_TFIDF.csv")

TARGET_COL = "priority_score"
QID_COL = "project_id"

df = df.sort_values(by=QID_COL).reset_index(drop=True)

print(f"Total Sampel Requirement : {len(df)}")
print(f"Total Jumlah Proyek (QID) : {df[QID_COL].nunique()}")
print(f"Daftar Proyek             : {df[QID_COL].unique()}")

df.head()


Total Sampel Requirement : 698
Total Jumlah Proyek (QID) : 15
Daftar Proyek             : ['P1' 'P10' 'P11' 'P12' 'P13' 'P14' 'P15' 'P2' 'P3' 'P4' 'P5' 'P6' 'P7'
 'P8' 'P9']


,id,project_id,type,value,effort,risk,stakeholder_priority,priority_score,rank,cleaned_requirement_text,...,tfidf_work,tfidf_workflow,tfidf_worklist,tfidf_workshop,tfidf_workstation,tfidf_xml,tfidf_year,tfidf_yet,tfidf_zero,tfidf_zipcode
0,REQ-01,P1,FR,-0.848439,-1.015393,-1.393797,-0.8401,1.3,18,head admin finance edit delete spare part cate...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,REQ-505,P1,NFR,-0.848439,0.165847,-0.550024,-0.8401,1.4,16,cosign note record date time signature,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,REQ-506,P1,NFR,0.444597,0.165847,1.981294,-0.8401,1.2,25,record display identity addended correct note ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,REQ-507,P1,NFR,-0.848439,0.165847,1.981294,-0.8401,0.8,33,record patient specific instruction educationa...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,REQ-508,P1,NFR,-0.848439,0.165847,1.981294,-0.8401,0.8,33,record date stamp prescription event initial c...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Step 3 | Group Based Data Splitting

In [3]:
# 1. Pisahkan Query ID dan Target
qids = df[QID_COL]
y = (df[TARGET_COL] * 10).astype(int)  # Ubah float (1.5) jadi int (15) untuk Ranker

# 2. Hapus kolom metadata & target dari matriks fitur X
COLS_TO_DROP = [
    "id",                       # ID requirement
    "type",                     # Jenis requirement (FR/NFR)
    "rank",                     # WAJIB DROP: Jangan sampai terbawa (data leakage)
    "cleaned_requirement_text", # Teks mentah
    TARGET_COL,                 # priority_score
    QID_COL                     # project_id
]

X = df.drop(columns=COLS_TO_DROP)

# 3. GroupShuffleSplit (80% Train, 20% Test berdasarkan kelompok project_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=qids))

# 4. Potong Data Train & Test
X_train, y_train, qids_train = X.iloc[train_idx].copy(), y.iloc[train_idx].copy(), qids.iloc[train_idx].copy()
X_test, y_test, qids_test = X.iloc[test_idx].copy(), y.iloc[test_idx].copy(), qids.iloc[test_idx].copy()

# 5. Urutkan runtut berdasarkan project_id
train_order = np.argsort(qids_train.values)
X_train = X_train.iloc[train_order].reset_index(drop=True)
y_train = y_train.iloc[train_order].reset_index(drop=True)
qids_train = qids_train.iloc[train_order].reset_index(drop=True)

test_order = np.argsort(qids_test.values)
X_test = X_test.iloc[test_order].reset_index(drop=True)
y_test = y_test.iloc[test_order].reset_index(drop=True)
qids_test = qids_test.iloc[test_order].reset_index(drop=True)

# 6. Hitung struktur ukuran grup per proyek
group_train = qids_train.value_counts(sort=False).values
group_test = qids_test.value_counts(sort=False).values

print(f"Data Train : {len(X_train)} requirement dari {len(group_train)} proyek")
print(f"Data Test  : {len(X_test)} requirement dari {len(group_test)} proyek")
print(f"Jumlah Fitur X : {X_train.shape[1]} (Domain features [value, effort, risk, stakeholder] + TF-IDF)")


Data Train : 453 requirement dari 12 proyek
Data Test  : 245 requirement dari 3 proyek
Jumlah Fitur X : 1431 (Domain features [value, effort, risk, stakeholder] + TF-IDF)


#### Step 4 | Evaluasi Rangking (NDCG, MAP, Spearman, Kendall)

In [4]:
def compute_map(y_true, y_pred, threshold=None):
    """Menghitung Mean Average Precision (MAP) per kelompok."""
    if threshold is None:
        threshold = np.median(y_true)
    
    binary_true = (y_true >= threshold).astype(int)
    if binary_true.sum() == 0:
        return 0.0
    
    # Sort berdasarkan prediksi berurutan menurun (descending)
    order = np.argsort(-y_pred)
    binary_true = binary_true[order]
    
    precisions = []
    n_relevant = 0
    for i, rel in enumerate(binary_true):
        if rel == 1:
            n_relevant += 1
            precisions.append(n_relevant / (i + 1))
            
    return np.mean(precisions) if precisions else 0.0

def evaluate_ranking_metrics(y_true, y_pred, qids, k=5):
    """Evaluasi korelasi urutan dan mutu ranking secara rata-rata di seluruh query group."""
    spearman_list, kendall_list, ndcg_list, map_list = [], [], [], []
    
    for qid in np.unique(qids):
        mask = (qids == qid)
        yt = y_true[mask].values
        yp = y_pred[mask]
        
        if len(yt) > 1:
            # Spearman & Kendall Tau
            rh, _ = spearmanr(yt, yp)
            kt, _ = kendalltau(yt, yp)
            if not np.isnan(rh): spearman_list.append(rh)
            if not np.isnan(kt): kendall_list.append(kt)
            
            # NDCG_K
            ndcg = ndcg_score([yt], [yp], k=k)
            ndcg_list.append(ndcg)
            
            # MAP
            map_val = compute_map(yt, yp)
            map_list.append(map_val)
            
    return {
        "Spearman": np.mean(spearman_list) if spearman_list else 0.0,
        "Kendall": np.mean(kendall_list) if kendall_list else 0.0,
        f"NDCG_{k}": np.mean(ndcg_list) if ndcg_list else 0.0,
        "MAP": np.mean(map_list) if map_list else 0.0
    }


#### Step 5 | Pairwise Data Construction (SVMRank)

In [5]:
def create_pairwise_data(X_data, y_data, qids_data):
    """Mengubah dataset pointwise menjadi pasangan perbedaan fitur (Pairwise transformation untuk SVMRank)."""
    X_pairs, y_pairs = [], []
    
    for qid in np.unique(qids_data):
        idx = np.where(qids_data == qid)[0]
        X_group = X_data.iloc[idx].values
        y_group = y_data.iloc[idx].values
        
        n_samples = len(idx)
        for i in range(n_samples):
            for j in range(i + 1, n_samples):
                if y_group[i] != y_group[j]:
                    # Selisih fitur
                    diff = X_group[i] - X_group[j]
                    label = 1 if y_group[i] > y_group[j] else -1
                    
                    X_pairs.append(diff)
                    y_pairs.append(label)
                    
                    # Tambahkan simetri sebaliknya untuk menyeimbangkan kelas (+1 dan -1)
                    X_pairs.append(-diff)
                    y_pairs.append(-label)
                    
    return np.array(X_pairs), np.array(y_pairs)

print("Helper Pairwise SVMRank siap digunakan.")


Helper Pairwise SVMRank siap digunakan.


#### Step 6 | Training 4 Algoritma

In [ ]:
import joblib

# Inisialisasi dictionary hasil & dataframe penampung prediksi
results_summary = {}
trained_models = {}
df_preds = df.iloc[test_idx][["id", "project_id", "priority_score"]].iloc[test_order].copy().reset_index(drop=True)

print("Penampung evaluasi dan prediksi siap.")


##### Step 6.1 | Training LightGBM Ranker

In [ ]:
model_name = "LightGBM_Ranker"
with mlflow.start_run(run_name=model_name):
    print(f"=================== Training {model_name} ===================")
    model = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(X_train, y_train, group=group_train)
    preds = model.predict(X_test)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = model
    
    if hasattr(model, 'get_params'):
        mlflow.log_params(model.get_params())
    
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> {m_name}: {m_val:.4f}")
        
    mlflow.lightgbm.log_model(model, artifact_path="model", serialization_format="cloudpickle")
    print()


##### Step 6.2 | Training XGBoost Ranker

In [ ]:
model_name = "XGBoost_Ranker"
with mlflow.start_run(run_name=model_name):
    print(f"=================== Training {model_name} ===================")
    model = xgb.XGBRanker(
        objective="rank:ndcg",
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(X_train, y_train, group=group_train)
    preds = model.predict(X_test)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = model
    
    if hasattr(model, 'get_params'):
        mlflow.log_params(model.get_params())
    
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> {m_name}: {m_val:.4f}")
        
    mlflow.xgboost.log_model(model, artifact_path="model")
    print()


##### Step 6.3 | Training SVMRank

In [ ]:
model_name = "SVMRank"
with mlflow.start_run(run_name=model_name):
    print(f"=================== Training {model_name} ===================")
    X_train_p, y_train_p = create_pairwise_data(X_train, y_train, qids_train)
    model = LinearSVC(C=1.0, random_state=42, max_iter=2000)
    model.fit(X_train_p, y_train_p)
    preds = model.decision_function(X_test)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = model
    
    if hasattr(model, 'get_params'):
        mlflow.log_params(model.get_params())
    
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> {m_name}: {m_val:.4f}")
        
    mlflow.sklearn.log_model(model, artifact_path="model", serialization_format="cloudpickle")
    print()


##### Step 6.4 | Training Random Forest

In [ ]:
model_name = "Random_Forest"
with mlflow.start_run(run_name=model_name):
    print(f"=================== Training {model_name} ===================")
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = model
    
    if hasattr(model, 'get_params'):
        mlflow.log_params(model.get_params())
    
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> {m_name}: {m_val:.4f}")
        
    mlflow.sklearn.log_model(model, artifact_path="model", serialization_format="cloudpickle")
    print()


#### Step 7 | Export Results

In [ ]:
import os
import joblib

# 1. Buat folder results dan subfolder models jika belum ada
os.makedirs("results/models", exist_ok=True)

# 2. Simpan Ringkasan Perbandingan Metrik ke CSV
df_results = pd.DataFrame(results_summary).T
df_results.to_csv("results/phase_2_model_comparison.csv", index_label="Model")

print("=== HASIL EKSPERIMEN PHASE 2: RANKING ALGORITHM COMPARISON ===")
display(df_results)

# 3. Simpan Detail Prediksi Test Set ke CSV
df_preds.to_csv("results/phase_2_predictions.csv", index=False)

# 4. Simpan SEMUA model ke folder results/models/
print("\n💾 Menyimpan semua file pickle (.pkl) model ke 'results/models/':")
for name, m in trained_models.items():
    model_path = f"results/models/{name}.pkl"
    joblib.dump(m, model_path)
    print(f"   - Model '{name}' -> {model_path}")

# 5. Evaluasi & Simpan Model Terbaik ke results/best_ranking_model.pkl
best_model_name = df_results["NDCG_5"].idxmax()
best_score = df_results.loc[best_model_name, "NDCG_5"]

print(f"\n🏆 BEST RANKING ALGORITHM (Model Selection 2): {best_model_name}")
print(f"   Skor NDCG_5 Terbaik: {best_score:.4f}")

joblib.dump(trained_models[best_model_name], "results/best_ranking_model.pkl")

print("\n✅ Semua file hasil eksekusi berhasil disimpan di folder 'results/':")
print("   1. results/phase_2_model_comparison.csv")
print("   2. results/phase_2_predictions.csv")
print("   3. results/best_ranking_model.pkl")
print("   4. results/models/ (berisi 4 file .pkl untuk semua algoritma)")


### Summary Results

In [ ]:
import os
import joblib

# 1. Buat folder results dan subfolder models jika belum ada
os.makedirs("results/models", exist_ok=True)

# 2. Simpan Ringkasan Perbandingan Metrik ke CSV
df_results = pd.DataFrame(results_summary).T
df_results.to_csv("results/phase_2_model_comparison.csv", index_label="Model")

print("=== HASIL EKSPERIMEN PHASE 2: RANKING ALGORITHM COMPARISON ===")
display(df_results)

# 3. Simpan Detail Prediksi Test Set ke CSV
df_preds.to_csv("results/phase_2_predictions.csv", index=False)

# 4. Simpan SEMUA model ke folder results/models/
print("\n💾 Menyimpan semua file pickle (.pkl) model ke 'results/models/':")
for name, m in trained_models.items():
    model_path = f"results/models/{name}.pkl"
    joblib.dump(m, model_path)
    print(f"   - Model '{name}' -> {model_path}")

# 5. Evaluasi & Simpan Model Terbaik ke results/best_ranking_model.pkl
best_model_name = df_results["NDCG_5"].idxmax()
best_score = df_results.loc[best_model_name, "NDCG_5"]

print(f"\n🏆 BEST RANKING ALGORITHM (Model Selection 2): {best_model_name}")
print(f"   Skor NDCG_5 Terbaik: {best_score:.4f}")

joblib.dump(trained_models[best_model_name], "results/best_ranking_model.pkl")

print("\n✅ Semua file hasil eksekusi berhasil disimpan di folder 'results/':")
print("   1. results/phase_2_model_comparison.csv")
print("   2. results/phase_2_predictions.csv")
print("   3. results/best_ranking_model.pkl")
print("   4. results/models/ (berisi 4 file .pkl untuk semua algoritma)")
